## load_fema_nri
Loads the FEMA National Risk Index **county** table (`nri_counties.csv`, 467 source cols) from `RAW_FEMA_NRI` (`/Volumes/{CATALOG}/raw/fema_nri/`) into the all-STRING Bronze table `{BRONZE}.fema_nri_counties`. Only the curated **29** analytical columns are kept (4 identity + 5 composite + 10 hazards x {score, rating}); the full 467-col CSV remains the fidelity record in the Volume.

**Write strategy (A):** MERGE upsert on `stcofips` (one row per county). NRI is an annual single-vintage snapshot; re-running the same vintage is a no-op, a new vintage updates in place.

**No archiving (deliberate deviation from CLAUDE.md SS10 cell 7, recorded per SS18):** NRI is a full snapshot re-downloaded each cycle by the idempotent weather_download job. DDL: `libs/ddl/weather_data.py`. Design: `_dev_planning/design_docs/weather_bronze_load_design.md`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: BRONZE, AUDIT, RAW_FEMA_NRI, PIPELINE_RUN_ID, STATUS_*, StepLog,
# Utils, ingestion_log_insert, spark, dbutils, F.

STEP_SEQUENCE = 1                                   # position owned by the orchestrator
SOURCE_SYSTEM = "fema_nri"
SOURCE_PATH   = RAW_FEMA_NRI                        # /Volumes/{CATALOG}/raw/fema_nri/
TARGET_TABLE  = f"{BRONZE}.fema_nri_counties"

# The curated 29 source columns, in the source's UPPER case (the landed CSV has these among
# its 467). Single source of truth: cell 4 validates these are present, cell 5 selects them.
# Lowercasing each yields the DDL column names (weather_data.py); no other rename is needed.
REQUIRED_SOURCE_COLS = [
    "STCOFIPS", "COUNTY", "STATEABBRV", "POPULATION",
    "RISK_SCORE", "RISK_RATNG", "EAL_VALT", "SOVI_RATNG", "RESL_RATNG",
    "HRCN_RISKS", "HRCN_RISKR", "CFLD_RISKS", "CFLD_RISKR", "IFLD_RISKS", "IFLD_RISKR",
    "TRND_RISKS", "TRND_RISKR", "WFIR_RISKS", "WFIR_RISKR", "ERQK_RISKS", "ERQK_RISKR",
    "HAIL_RISKS", "HAIL_RISKR", "SWND_RISKS", "SWND_RISKR", "HWAV_RISKS", "HWAV_RISKR",
    "WNTW_RISKS", "WNTW_RISKR",
]

# MERGE natural key — one row per county (annual single-vintage snapshot).
MERGE_KEYS = ["stcofips"]

In [ ]:
# Open the pipeline_step_log row (RUNNING). Closed explicitly in cell 6 (succeed) or by
# step.fail(e) in any work cell's 2-line handler.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
)
print(f"load_fema_nri: step_log_id={step.step_log_id}")

In [ ]:
# Per-file header validation BEFORE the bulk read. NRI has 467 columns, of which we keep 29
# — so we validate the 29 required columns are a SUBSET of the header (not exact equality),
# failing on any that are missing. The no-files CHECK stays inside the try (a failed
# dbutils.fs.ls is logged via step.fail); the early EXIT goes OUTSIDE the try, because
# dbutils.notebook.exit() raises an ordinary exception that `except Exception` would swallow
# (no dbutils.NotebookExit class) — CLAUDE.md SS10.1.
no_files = False
try:
    files = [f.path for f in dbutils.fs.ls(SOURCE_PATH) if f.path.lower().endswith(".csv")]
    no_files = not files
    if not no_files:
        bad_files = []
        for file_path in files:
            actual_cols = (
                spark.read.format("csv").option("header", "true")
                .load(file_path).limit(0).columns
            )
            missing = [c for c in REQUIRED_SOURCE_COLS if c not in actual_cols]
            if missing:
                bad_files.append((file_path, missing))
        if bad_files:
            raise ValueError(
                f"[{TARGET_TABLE}] Required column(s) missing in {len(bad_files)} file(s).\n"
                + "\n".join(f"  {p}\n    missing: {m}" for p, m in bad_files)
            )
        print(f"load_fema_nri: {len(files)} file(s) passed header validation.")
except Exception as e:
    step.fail(e); raise

if no_files:
    step.no_files()
    dbutils.notebook.exit(f"No CSV files found at {SOURCE_PATH}")

In [ ]:
# Read the 467-col CSV all-STRING by NAME: header=true gives the column names, and
# inferSchema=false disables type inference so every column is STRING — this is NOT
# inferSchema (the opposite), and is more robust than a positional schema for 467 cols.
# Select the 29 by name and lowercase to the DDL column names, then add the audit columns
# (executor-consistent current_timestamp, never datetime.now; run_id as STRING per SS18).
try:
    raw_df = (
        spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(SOURCE_PATH)
            .withColumn("source_file_path", F.col("_metadata.file_path"))
    )
    selected_cols = [F.col(source_col).alias(source_col.lower()) for source_col in REQUIRED_SOURCE_COLS]
    shaped_df = (
        raw_df
            .select(*selected_cols, "source_file_path")
            .withColumn("inserted_ts", F.current_timestamp())
            .withColumn("run_id", F.lit(PIPELINE_RUN_ID))
    )
    rows_read = shaped_df.count()
    step.rows_read = rows_read
    print(f"load_fema_nri: read {rows_read:,} rows from {SOURCE_PATH}")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Strategy A: MERGE upsert on the natural key. INSERT */UPDATE SET * match by column name
# (staging columns are exactly the target columns). MERGE metrics (num_inserted_rows /
# num_updated_rows) come back as the result row on Databricks [Projected — confirm on first
# run]; fall back to the post-pre delta for the insert count if absent.
try:
    shaped_df.createOrReplaceTempView("fema_nri_staging")
    on_clause = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEYS)

    pre_count = spark.table(TARGET_TABLE).count()
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS t
        USING fema_nri_staging AS s
        ON {on_clause}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first().asDict()
    post_count = spark.table(TARGET_TABLE).count()

    inserted = metrics.get("num_inserted_rows")
    updated  = metrics.get("num_updated_rows")
    if inserted is None:
        inserted = post_count - pre_count
    if post_count - pre_count != inserted:
        raise AssertionError(
            f"[{TARGET_TABLE}] Insert-count mismatch: MERGE reported {inserted:,} inserts, "
            f"but row count grew by {post_count - pre_count:,} "
            f"(pre {pre_count:,}, post {post_count:,})."
        )

    step.rows_written = inserted
    step.succeed()
    print(f"load_fema_nri: MERGE done — inserted={inserted:,} updated={updated} "
          f"(read={rows_read:,}, pre={pre_count:,}, post={post_count:,}).")
except Exception as e:
    step.fail(e); raise

# ingestion_log (leaf tier) runs AFTER succeed() and OUTSIDE the write try: the helper
# swallows its own errors and returns a dict, so a logging hiccup cannot roll back the
# committed MERGE (CLAUDE.md SS11.4). One row per ingested file.
files_df = shaped_df.select("source_file_path").distinct()
res = ingestion_log_insert(
    spark, AUDIT, files_df, PIPELINE_RUN_ID, step.step_log_id,
    source_system=SOURCE_SYSTEM, target_table=TARGET_TABLE,
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"load_fema_nri: WARNING ingestion_log insert failed: {res['error_message']}")